# Removing Rejected Trials

In [ ]:
from scipy.io import loadmat
import os

In [ ]:
TRIAL_DURATION = 7.5  # seconds[cite: 5]
N_EEG_CH = 22  # 22 EEG channels; exclude EOG per rules[cite: 5]

In [ ]:
def load_clean_subject_mat(subject=1, session="T"):
    paths = BNCI2014_001().data_path(subject)
    mat_by_sess = {os.path.basename(p)[-5]: p for p in paths}
    mat = loadmat(mat_by_sess[session], squeeze_me=True, struct_as_record=False)

    clean_X, clean_y = [], []
    for idx, run in enumerate(mat["data"]):
        trl = np.asarray(run.trial, dtype=int).ravel()
        if trl.size == 0:
            continue

        fs = float(np.asarray(run.fs).item()) if hasattr(run, "fs") else 250.0
        n_samples = int(round(TRIAL_DURATION * fs))
        X_eeg = np.asarray(run.X)[:, :N_EEG_CH]
        y_arr = np.asarray(run.y).ravel()
        art = np.asarray(run.artifacts).ravel()

        keep_idx = np.where(art == 0)[0]
        print(f"run {idx}: total={len(trl)}, rejected={(art == 1).sum()}, kept={len(keep_idx)}")

        for t_i in keep_idx:
            s0 = int(trl[t_i])
            s1 = s0 + n_samples
            if 0 <= s0 and s1 <= X_eeg.shape[0]:
                clean_X.append(X_eeg[s0:s1].T)  # (22, 1875)
                clean_y.append(y_arr[t_i])

    X_clean = np.stack(clean_X, axis=0) if clean_X else np.empty((0, N_EEG_CH, 0))
    y_clean = np.asarray(clean_y)
    return X_clean, y_clean

In [ ]:
X_s1, y_s1 = load_clean_subject_mat(subject=1, session="T")
print(f"Subject 1 clean shape: {X_s1.shape}, labels shape: {y_s1.shape}")